## 알렉스넷 모델 아키텍처
파이토치 버전 알렉스넷

In [ ]:
!pip install torchinfo

In [ ]:

import torch
import torch.nn as nn

class AlexNet(nn.Module):
    def __init__(self, num_classes=1000):
        super(AlexNet, self).__init__()
        
        self.features = nn.Sequential(
            # Conv1 (conv + pool + batchnorm)
            # Conv2d: H_out = floor((H_in + 2*padding - kernel_size) / stride) + 1
            nn.Conv2d(in_channels=3, out_channels=96, kernel_size=11, stride=4, padding=0),         # (3,227,227)→(96,55,55): (227-11)/4+1=55, 채널 3→96
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=0),   # (96,55,55)→(96,27,27): (55-3)/2+1=27, 채널 불변
            nn.BatchNorm2d(96),                                 # (96,27,27)→(96,27,27): shape 불변, 96채널 각각 정규화
            
            # Conv2 (conv + pool + batchnorm)
            nn.Conv2d(in_channels=96, out_channels=256, kernel_size=5, stride=1, padding=2), # (96,27,27)→(256,27,27): padding=2로 H유지, 채널 96→256
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=0),   # (256,27,27)→(256,13,13): (27-3)/2+1=13, 채널 불변
            nn.BatchNorm2d(256),                                # (256,13,13)→(256,13,13): shape 불변, 256채널 각각 정규화
            
            # Conv3 (conv + batchnorm)
            nn.Conv2d(in_channels=256, out_channels=384, kernel_size=3, stride=1, padding=1), # (256,13,13)→(384,13,13): padding=1로 H유지, 채널 256→384
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(384),                                 # (384,13,13)→(384,13,13): shape 불변, 384채널 각각 정규화
            
            # Conv4 (conv + batchnorm)
            nn.Conv2d(in_channels=384, out_channels=384, kernel_size=3, stride=1, padding=1), # (384,13,13)→(384,13,13): padding=1로 H유지, 채널 불변
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(384),                                 # (384,13,13)→(384,13,13): shape 불변, 384채널 각각 정규화
            
            # Conv5 (conv + batchnorm)
            nn.Conv2d(in_channels=384, out_channels=256, kernel_size=3, stride=1, padding=1), # (384,13,13)→(256,13,13): padding=1로 H유지, 채널 384→256
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(256),                                 # (256,13,13)→(256,13,13): shape 불변, 256채널 각각 정규화
            nn.MaxPool2d(kernel_size=3, stride=2, padding=0)    # (256,13,13)→(256,6,6): (13-3)/2+1=6, 채널 불변
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            
            # FC6 (Dense layer + dropout)
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            
            # FC7 (Dense layers)
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            
            # FC8 (softmax output layer)
            nn.Linear(4096, num_classes),
            nn.Softmax(dim=1)  
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# 모델 객체 생성 및 요약 출력
model = AlexNet()

from torchinfo import summary
summary(model, input_size=(1, 3, 227, 227))


Layer (type:depth-idx)                   Output Shape              Param #
AlexNet                                  [1, 1000]                 --
├─Sequential: 1-1                        [1, 256, 6, 6]            --
│    └─Conv2d: 2-1                       [1, 96, 55, 55]           34,944
│    └─ReLU: 2-2                         [1, 96, 55, 55]           --
│    └─MaxPool2d: 2-3                    [1, 96, 27, 27]           --
│    └─BatchNorm2d: 2-4                  [1, 96, 27, 27]           192
│    └─Conv2d: 2-5                       [1, 256, 27, 27]          614,656
│    └─ReLU: 2-6                         [1, 256, 27, 27]          --
│    └─MaxPool2d: 2-7                    [1, 256, 13, 13]          --
│    └─BatchNorm2d: 2-8                  [1, 256, 13, 13]          512
│    └─Conv2d: 2-9                       [1, 384, 13, 13]          885,120
│    └─ReLU: 2-10                        [1, 384, 13, 13]          --
│    └─BatchNorm2d: 2-11                 [1, 384, 13, 13]          76

케라스 버전 알렉스넷

In [6]:
import tensorflow
from keras.models import Sequential
from keras.layers import Dense, Dropout, Conv2D, MaxPool2D, Flatten, AveragePooling2D, Activation
from keras.layers import BatchNormalization
from keras.regularizers import l2

model = Sequential(name="Alexnet")
# Conv1 (conv + pool + batchnorm)
model.add(Conv2D(filters= 96, kernel_size=(11,11), strides=(4,4), padding='valid', kernel_regularizer=l2(0.0005), input_shape = (227,227,3)))
model.add(Activation('relu')) 
model.add(MaxPool2D(pool_size=(3,3), strides= (2,2), padding='valid'))   # pool 5x5
model.add(BatchNormalization())
    
# Conv2 (conv + pool + batchnorm)
model.add(Conv2D(filters=256, kernel_size=(5,5), strides=(1,1), padding='same', kernel_regularizer=l2(0.0005)))
model.add(Activation('relu'))
model.add(MaxPool2D(pool_size=(3,3), strides=(2,2), padding='valid'))
model.add(BatchNormalization())
            
# Conv3 (conv + batchnorm)      
model.add(Conv2D(filters=384, kernel_size=(3,3), strides=(1,1), padding='same', kernel_regularizer=l2(0.0005)))
model.add(Activation('relu'))
model.add(BatchNormalization())
        
# Conv4 (conv + batchnorm)
model.add(Conv2D(filters=384, kernel_size=(3,3), strides=(1,1), padding='same', kernel_regularizer=l2(0.0005)))
model.add(Activation('relu'))
model.add(BatchNormalization())
            
# Conv5 (conv + batchnorm)  
model.add(Conv2D(filters=256, kernel_size=(3,3), strides=(1,1), padding='same', kernel_regularizer=l2(0.0005)))
model.add(Activation('relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(3,3), strides=(2,2), padding='valid'))
model.add(Flatten())

# FC6 (Dense layer + dropout)  
model.add(Dense(units = 4096, activation = 'relu'))
model.add(Dropout(0.5))

# FC7 (Dense layers) 
model.add(Dense(units = 4096, activation = 'relu'))
model.add(Dropout(0.5))
                           
# FC8 (softmax output layer) 
model.add(Dense(units = 1000, activation = 'softmax'))

model.summary()



c:\ProgramData\miniconda3\envs\venv_lmm\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "Alexnet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 55, 55, 96)     │        34,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 55, 55, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 27, 27, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 27, 27, 96)     │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 27, 27, 256)    │       614,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 27, 27, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 13, 13, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 13, 13, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 13, 13, 384)    │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 13, 13, 384)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 13, 13, 384)    │         1,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 13, 13, 384)    │     1,327,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 13, 13, 384)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 13, 13, 384)    │         1,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 13, 13, 256)    │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 13, 13, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 13, 13, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4096)           │    37,752,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 62,383,848 (237.98 MB)

 Trainable params: 62,381,096 (237.96 MB)

 Non-trainable params: 2,752 (10.75 KB)

각 버전별 학습 파라메터(Trainable params) 62,381,096개 일치